<a href="https://colab.research.google.com/github/koji-mikajiri/cartpole-practice-2026-05-31/blob/main/cartpole_practice_2026_05_31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gymnasium

In [2]:
import gymnasium as gym

# 環境初期化
env = gym.make("CartPole-v1")

# 環境リセット
observation, info = env.reset()

  # タイムステップのループ
for step in range(100):

    action = env.action_space.sample()


    observation, reward, terminated, truncated, info = env.step(action)

    print(f"Step: {step}, Action: {action}, Reward: {reward}")

    # 終了チェック
    if terminated or truncated:
        print("終了。環境をリセットします。")
        observation, info = env.reset()

# 閉じる
env.close()

Step: 0, Action: 1, Reward: 1.0
Step: 1, Action: 1, Reward: 1.0
Step: 2, Action: 0, Reward: 1.0
Step: 3, Action: 0, Reward: 1.0
Step: 4, Action: 0, Reward: 1.0
Step: 5, Action: 1, Reward: 1.0
Step: 6, Action: 0, Reward: 1.0
Step: 7, Action: 0, Reward: 1.0
Step: 8, Action: 1, Reward: 1.0
Step: 9, Action: 1, Reward: 1.0
Step: 10, Action: 0, Reward: 1.0
Step: 11, Action: 1, Reward: 1.0
Step: 12, Action: 1, Reward: 1.0
Step: 13, Action: 0, Reward: 1.0
Step: 14, Action: 0, Reward: 1.0
Step: 15, Action: 0, Reward: 1.0
Step: 16, Action: 0, Reward: 1.0
Step: 17, Action: 1, Reward: 1.0
Step: 18, Action: 1, Reward: 1.0
Step: 19, Action: 0, Reward: 1.0
Step: 20, Action: 0, Reward: 1.0
Step: 21, Action: 1, Reward: 1.0
Step: 22, Action: 0, Reward: 1.0
Step: 23, Action: 0, Reward: 1.0
Step: 24, Action: 1, Reward: 1.0
Step: 25, Action: 0, Reward: 1.0
Step: 26, Action: 0, Reward: 1.0
Step: 27, Action: 1, Reward: 1.0
Step: 28, Action: 1, Reward: 1.0
Step: 29, Action: 0, Reward: 1.0
Step: 30, Action: 1,

In [3]:
# 状況把握するためのobservationを置く

import gymnasium as gym

env = gym.make("CartPole-v1")
observation, info = env.reset()

print("--- 状態（Observation）の初期値 ---")
print(observation)

print("\n--- 状態空間（Observation Space）の詳細 ---")
print(f"データの形: {env.observation_space.shape}")
print(f"最小値: {env.observation_space.low}")
print(f"最大値: {env.observation_space.high}")

env.close()



# わかりやすい例え（Geminiによる生成）：「人間が指の上でホウキを立ててバランスを取る時、「ホウキが右に傾いてるな」「勢いよく倒れそうだな」と目で見て手を動かしますよね。AIにとっての「目」が、まさにこの4つの数値です。」

--- 状態（Observation）の初期値 ---
[ 0.01794234 -0.04870273 -0.03817021 -0.02100563]

--- 状態空間（Observation Space）の詳細 ---
データの形: (4,)
最小値: [-4.8               -inf -0.41887903        -inf]
最大値: [4.8               inf 0.41887903        inf]


In [4]:
# PyTorchを使用
import torch
import torch.nn as nn

# 4つの入力（状態）を受け取り、2つの出力（行動の価値）を返すニューラルネットワークをつくる
class SimpleQNetwork(nn.Module):
    def __init__(self):
        super(SimpleQNetwork, self).__init__()
        # 入力層(4) -> 隠れ層(128)
        self.fc1 = nn.Linear(4, 128)
        # 隠れ層(128) -> 出力層(2)
        self.fc2 = nn.Linear(128, 2)
        # 活性化関数（ReLU）
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# ネットワークを実体化（インスタンス化）
model = SimpleQNetwork()
print("--- 構築したニューラルネットワーク ---")
print(model)


# テスト：さきほどのObservation（状態）をネットワークに渡してみる

# NumPy配列をPyTorchの「テンソル」というデータ形式に変換
obs_tensor = torch.FloatTensor(observation)

# ネットワークに入力して出力を得る（順伝播：Forward）

# メモ：「スコア」は、強化学習（Q学習）において「Q値」と呼ばれ、「その行動をとった時に将来どれくらい報酬がもらえそうかの期待値」
output = model(obs_tensor)

print("\n--- AIの出力（各行動の予測スコア） ---")
print(f"行動0（左へ押す）のスコア: {output[0].item():.4f}")
print(f"行動1（右へ押す）のスコア: {output[1].item():.4f}")


# この段階ではまだ強化学習はしていない

--- 構築したニューラルネットワーク ---
SimpleQNetwork(
  (fc1): Linear(in_features=4, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
  (relu): ReLU()
)

--- AIの出力（各行動の予測スコア） ---
行動0（左へ押す）のスコア: 0.2125
行動1（右へ押す）のスコア: -0.0752


In [5]:
# 一個前のセルでつくったニューラルネットワークをつかい、状態に応じた行動を賢く（一部ランダムに）選ぶ関数をつくる

import numpy as np
import random
import torch

# 行動を選ぶ関数
def select_action(state, epsilon):
    # 確率 epsilon でランダムに行動を選択（探索）
    if random.random() < epsilon:
        return env.action_space.sample()

    # それ以外は、AIの予測（Q値）が一番高い行動を選択（利用）
    else:
        # 勾配計算を無効化して高速化
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state)
            q_values = model(state_tensor)

            # torch.no_grad()の意味: 今は「予測」をするだけなので、バックプロパゲーション用の無駄な計算をオフにしてスピードアップ


            # AIが出力した2つのスコアのうち、数値が大きい方の位置（0か1）を抜き
            return torch.argmax(q_values).item()

# テスト用の設定
epsilon = 0.1  # 10%の確率でランダム、90%の確率でAIの判断
current_state = observation  # さきほどの初期状態

# 関数を試してみる
chosen_action = select_action(current_state, epsilon)
print(f"現在の状態において、AIが選択した行動: {chosen_action}")

現在の状態において、AIが選択した行動: 0


In [6]:
# PyTorchの機能を使って、「今もらえる報酬」＋「次の状態で期待できる最大スコア」を、現在の行動の正しい評価（正解）として扱い、AIの現在の予測をそこに近づけていく工程の「1ステップ分の学習」を行う
# ベルマン方程式については要復習

import torch.optim as optim

# どれくらいの歩幅でパラメータを修正するか
learning_rate = 0.01
# 報酬をどれくらい重視するか
gamma = 0.99

# 最適化手法（Adam）と損失関数（平均二乗誤差）を定義
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()



# 1ステップだけの疑似学習テスト


# 1. 現在の状態から行動を選択（先ほど作った関数を使用）
action = select_action(current_state, epsilon=0.1)

# 2. 環境に行動を適用し、結果を受け取る
next_state, reward, terminated, truncated, _ = env.step(action)

# 3. ターゲット（正解）のQ値を計算
with torch.no_grad():
    next_state_tensor = torch.FloatTensor(next_state)

    # 次の状態で一番高いQ値を取得
    next_max_q = torch.max(model(next_state_tensor)).item()

    # 終了状態なら未来の報酬はないので、現在の報酬だけにする
    if terminated or truncated:
        target_q = reward
    else:
        target_q = reward + gamma * next_max_q

# 4. 現在のネットワークの予測Q値を計算
current_state_tensor = torch.FloatTensor(current_state)
predict_q_values = model(current_state_tensor)
predict_q = predict_q_values[action] # 選んだ行動のQ値だけを取り出す

# 5. 予測とターゲットのズレ（Loss）を計算
target_q_tensor = torch.FloatTensor([target_q])
loss = loss_fn(predict_q, target_q_tensor)

# 6. ネットワークのパラメータを更新（誤差逆伝播法）
optimizer.zero_grad() # 前回の計算のゴミの部分（勾配）をリセット
loss.backward()       # 誤差をネットワーク全体に逆伝播して計算
optimizer.step()      # パラメータを少しだけ修正して賢くする

print(f"選んだ行動: {action}")
print(f"予測Q値: {predict_q.item():.4f}, ターゲットQ値: {target_q:.4f}")
print(f"Loss（誤差）: {loss.item():.4f}")
print("パラメータの1ステップ更新が完了しました！")

選んだ行動: 0
予測Q値: 0.2125, ターゲットQ値: 1.1852
Loss（誤差）: 0.9462
パラメータの1ステップ更新が完了しました！


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [7]:
# DQNなどについては現時点でまだ理解が追いついていないので手を出さない
# 完璧ではないが、とりあえず動いて徐々に学習していく全体像を体験することを優先した



# ここまでのセルのかく関数を統合した学習の全体ループをつくる
import gymnasium as gym
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 環境とネットワークの準備
env = gym.make("CartPole-v1")

class SimpleQNetwork(nn.Module):
    def __init__(self):
        super(SimpleQNetwork, self).__init__()
        self.fc1 = nn.Linear(4, 128)
        self.fc2 = nn.Linear(128, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleQNetwork()



# 2. 学習の設定
learning_rate = 0.01
gamma = 0.99
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()


# 探索率（最初は100%ランダム、徐々に減らしていく）
epsilon = 1.0
epsilon_decay = 0.99 # エピソードごとにランダム度合いを減らす
epsilon_min = 0.01   # 最低でも1%はランダムに行動する

num_episodes = 300   # 学習するエピソード数
reward_history = []  # スコアの記録用


# 3. 行動選択関数
def select_action(state, eps):
    if random.random() < eps:
        return env.action_space.sample()
    else:
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state)
            q_values = model(state_tensor)
            return torch.argmax(q_values).item()

# 4. 学習のメインループ
print("学習を開始します！...")

for episode in range(num_episodes):
    state, info = env.reset()
    total_reward = 0

    for step in range(500): # CartPoleの最大ステップは500
        # 行動を選ぶ
        action = select_action(state, epsilon)

        # 環境を進める
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward

        # --- ここからが学習（脳みその更新） ---
        with torch.no_grad():
            next_state_tensor = torch.FloatTensor(next_state)
            next_max_q = torch.max(model(next_state_tensor)).item()



            # 【重要】最低限の学習を成立させるための工夫を行う（Geminiによる提案）
            # 途中で倒れてしまった場合は「ダメだった」と教えるためにペナルティ(-1.0)を与える
            if terminated:
                target_q = -1.0
            elif truncated:
                target_q = reward
            else:
                target_q = reward + gamma * next_max_q

        state_tensor = torch.FloatTensor(state)
        predict_q_values = model(state_tensor)
        predict_q = predict_q_values[action]

        target_q_tensor = torch.FloatTensor([target_q])
        loss = loss_fn(predict_q, target_q_tensor)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 学習おわり



        # 状態を次のステップへ引き継ぐ
        state = next_state

        # エピソードが終了したらループを抜ける
        if terminated or truncated:
            break

    # スコアを記録
    reward_history.append(total_reward)


    # 探索率を少し下げる（ランダム行動を減らす）
    epsilon = max(epsilon_min, epsilon * epsilon_decay)


    # 10エピソードごとに進捗を表示させる
    if (episode + 1) % 10 == 0:
        recent_average = np.mean(reward_history[-10:])
        print(f"Episode: {episode + 1:3d}, 最近10回の平均スコア: {recent_average:5.1f}, ランダム率(Epsilon): {epsilon:.2f}")


print("学習が完了しました！")
env.close()



学習を開始します！...
Episode:  10, 最近10回の平均スコア:  20.6, ランダム率(Epsilon): 0.90
Episode:  20, 最近10回の平均スコア:  21.0, ランダム率(Epsilon): 0.82
Episode:  30, 最近10回の平均スコア:  18.0, ランダム率(Epsilon): 0.74
Episode:  40, 最近10回の平均スコア:  19.4, ランダム率(Epsilon): 0.67
Episode:  50, 最近10回の平均スコア:  18.0, ランダム率(Epsilon): 0.61
Episode:  60, 最近10回の平均スコア:  23.0, ランダム率(Epsilon): 0.55
Episode:  70, 最近10回の平均スコア:  19.7, ランダム率(Epsilon): 0.49
Episode:  80, 最近10回の平均スコア:  25.3, ランダム率(Epsilon): 0.45
Episode:  90, 最近10回の平均スコア:  22.3, ランダム率(Epsilon): 0.40
Episode: 100, 最近10回の平均スコア:  43.6, ランダム率(Epsilon): 0.37
Episode: 110, 最近10回の平均スコア:  21.3, ランダム率(Epsilon): 0.33
Episode: 120, 最近10回の平均スコア:  26.0, ランダム率(Epsilon): 0.30
Episode: 130, 最近10回の平均スコア:  31.5, ランダム率(Epsilon): 0.27
Episode: 140, 最近10回の平均スコア:  38.2, ランダム率(Epsilon): 0.24
Episode: 150, 最近10回の平均スコア:  30.6, ランダム率(Epsilon): 0.22
Episode: 160, 最近10回の平均スコア:  50.5, ランダム率(Epsilon): 0.20
Episode: 170, 最近10回の平均スコア:  26.6, ランダム率(Epsilon): 0.18
Episode: 180, 最近10回の平均スコア:  46.9, ランダム率(Epsilon): 0.